# 01 — Data cleaning and engineering
Lisbon Airbnb Market Intelligence | Expernetic Data Engineer Intern Assignment

Purpose: Cleaning the data and data engineering

In [4]:
# 02_data_cleaning_engineering.ipynb

# Lisbon Airbnb Market Intelligence

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
project_root = Path.cwd().parent
RAW = project_root / "data" / "raw"
PROCESSED = project_root / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("="*60)
print("PHASE 2: DATA CLEANING & ENGINEERING")
print("="*60)

# ============================================
# 1. LOAD DATA
# ============================================
print("\n[1] Loading raw data...")

listings = pd.read_csv(RAW / "listings.csv", low_memory=False)
calendar = pd.read_csv(RAW / "calendar.csv.gz", compression="gzip")
reviews = pd.read_csv(RAW / "reviews.csv", parse_dates=['date'])
neighbourhoods = pd.read_csv(RAW / "neighbourhoods.csv")

print(f" Listings: {len(listings):,} rows, {len(listings.columns)} columns")
print(f" Calendar: {len(calendar):,} rows")
print(f" Reviews: {len(reviews):,} rows")
print(f" Neighbourhoods: {len(neighbourhoods):,} rows")



PHASE 2: DATA CLEANING & ENGINEERING

[1] Loading raw data...
 Listings: 24,876 rows, 90 columns
 Calendar: 9,092,881 rows
 Reviews: 1,886,932 rows
 Neighbourhoods: 134 rows


In [5]:
# ============================================
# 2. CLEAN LISTINGS DATA
# ============================================
print("\n[2] Cleaning listings data...")

# 2.1 Clean price - remove $, €, and , symbols
listings['price_clean'] = (
    listings['price']
    .astype(str)
    .str.replace(r'[$,€]', '', regex=True)  # Remove $, €, and commas
    .str.replace(',', '')                   # Remove thousands separators
    .str.strip()
)

# Convert to float, handle any remaining errors
listings['price_clean'] = pd.to_numeric(listings['price_clean'], errors='coerce')

print(f" Price converted: {listings['price_clean'].notna().sum():,} valid prices")
print(f"  - Min: ${listings['price_clean'].min():.2f}")
print(f"  - Max: ${listings['price_clean'].max():.2f}")
print(f"  - Mean: ${listings['price_clean'].mean():.2f}")

# 2.2 Handle missing bedrooms (17.92% null)
bedroom_median_by_type = listings.groupby('room_type')['bedrooms'].median()
listings['bedrooms_imputed'] = listings.apply(
    lambda row: row['bedrooms'] if pd.notna(row['bedrooms']) 
    else bedroom_median_by_type.get(row['room_type'], 1),
    axis=1
)
print(f" Bedrooms imputed: {listings['bedrooms'].isna().sum():,} null values filled")

# 2.3 Handle missing bathrooms (14.93% null)
bathroom_median_by_type = listings.groupby('room_type')['bathrooms'].median()
listings['bathrooms_imputed'] = listings.apply(
    lambda row: row['bathrooms'] if pd.notna(row['bathrooms']) 
    else bathroom_median_by_type.get(row['room_type'], 1),
    axis=1
)
print(f" Bathrooms imputed: {listings['bathrooms'].isna().sum():,} null values filled")

# 2.4 Create derived fields
# Host tenure (already in years from the data)
listings['host_tenure_years'] = listings['hosts_time_as_host_years'].fillna(0)

# Price per bedroom
listings['price_per_bedroom'] = listings['price_clean'] / listings['bedrooms_imputed']
listings['price_per_bedroom'] = listings['price_per_bedroom'].replace([np.inf, -np.inf], np.nan)

print(f" Created derived fields:")
print(f"  - host_tenure_years")
print(f"  - price_per_bedroom")




[2] Cleaning listings data...
 Price converted: 22,636 valid prices
  - Min: $3.67
  - Max: $16077.50
  - Mean: $189.79
 Bedrooms imputed: 4,458 null values filled
 Bathrooms imputed: 3,715 null values filled
 Created derived fields:
  - host_tenure_years
  - price_per_bedroom


In [6]:
# ============================================
# 3. PROCESS CALENDAR DATA
# ============================================
print("\n[3] Processing calendar data...")

# Convert date strings to datetime
calendar['date'] = pd.to_datetime(calendar['date'])

# Check if we have data for all 365 days
calendar_dates = calendar['date'].unique()
print(f" Calendar dates range: {calendar_dates.min()} to {calendar_dates.max()}")
print(f" Total unique dates: {len(calendar_dates)}")

# Aggregate calendar per listing
calendar_agg = calendar.groupby('listing_id').agg({
    'available': lambda x: (x == 't').sum(),  # Count available days
    'date': ['min', 'max']  # Get first and last date
}).reset_index()

# Fix column names (flatten MultiIndex)
calendar_agg.columns = ['listing_id', 'available_days', 'min_date', 'max_date']

# Calculate total days - now the subtraction works!
calendar_agg['total_days'] = (calendar_agg['max_date'] - calendar_agg['min_date']).dt.days + 1
calendar_agg['occupancy_rate'] = 1 - (calendar_agg['available_days'] / calendar_agg['total_days'])
calendar_agg['occupancy_rate'] = calendar_agg['occupancy_rate'].clip(0, 1)  # Ensure between 0-1

print(f" Calendar aggregated for {len(calendar_agg):,} listings")
print(f"  - Average occupancy: {calendar_agg['occupancy_rate'].mean():.2%}")
print(f"  - Average available days: {calendar_agg['available_days'].mean():.0f}")




[3] Processing calendar data...
 Calendar dates range: 2026-06-23 00:00:00 to 2027-07-02 00:00:00
 Total unique dates: 375
 Calendar aggregated for 24,912 listings
  - Average occupancy: 38.67%
  - Average available days: 224


In [7]:
# ============================================
# 4. PROCESS REVIEWS DATA
# ============================================
print("\n[4] Processing reviews data...")

# Get review summary per listing
reviews_agg = reviews.groupby('listing_id').agg({
    'id': 'count',           # Number of reviews
    'date': ['min', 'max']   # First and last review date
}).reset_index()

# Fix column names
reviews_agg.columns = ['listing_id', 'review_count', 'first_review', 'last_review']

# Convert dates
reviews_agg['first_review'] = pd.to_datetime(reviews_agg['first_review'])
reviews_agg['last_review'] = pd.to_datetime(reviews_agg['last_review'])

print(f" Reviews aggregated for {len(reviews_agg):,} listings")
print(f"  - Date range: {reviews_agg['first_review'].min()} to {reviews_agg['last_review'].max()}")




[4] Processing reviews data...
 Reviews aggregated for 21,794 listings
  - Date range: 2010-07-24 00:00:00 to 2026-07-02 00:00:00


In [8]:
# ============================================
# 5. ENRICH LISTINGS
# ============================================
print("\n[5] Enriching listings with calendar and review data...")

# Start with a copy of listings
enriched = listings.copy()

# Merge with calendar data
enriched = enriched.merge(
    calendar_agg[['listing_id', 'occupancy_rate', 'available_days']],
    left_on='id',
    right_on='listing_id',
    how='left'
)

# Merge with review data
enriched = enriched.merge(
    reviews_agg[['listing_id', 'review_count', 'first_review', 'last_review']],
    left_on='id',
    right_on='listing_id',
    how='left'
)

# Calculate estimated annual revenue
enriched['estimated_annual_revenue'] = (
    enriched['price_clean'] * 
    (365 - enriched['availability_365']) * 
    0.90  # Airbnb takes ~10% fee
)

print(f" Enriched data created with {len(enriched):,} rows")
print(f"  - Revenue available for {enriched['estimated_annual_revenue'].notna().sum():,} listings")
print(f"  - Average annual revenue: ${enriched['estimated_annual_revenue'].mean():,.2f}")




[5] Enriching listings with calendar and review data...
 Enriched data created with 24,876 rows
  - Revenue available for 22,636 listings
  - Average annual revenue: $20,893.16


In [10]:
# ============================================
# 6. CREATE FINAL DATASET (FIXED)
# ============================================
print("\n[6] Creating final dataset...")

# First, let's check what columns we actually have in enriched
print("Available columns in enriched:")
print([col for col in enriched.columns if 'review' in col.lower() or 'first' in col.lower() or 'last' in col.lower()])

# Check if first_review and last_review exist
if 'first_review' not in enriched.columns:
    print(" 'first_review' not found - creating from review data")
    # If not found, create from the original reviews_agg
    if 'first_review' in reviews_agg.columns:
        enriched = enriched.merge(
            reviews_agg[['listing_id', 'review_count', 'first_review', 'last_review']],
            left_on='id',
            right_on='listing_id',
            how='left',
            suffixes=('', '_reviews')
        )

# Now check again what we have
print("\nColumns after merge check:")
review_cols = [col for col in enriched.columns if 'review' in col.lower() or 'first' in col.lower() or 'last' in col.lower()]
print(review_cols)

# Define columns based on what's available
base_columns = [
    'id',                    # listing_id
    'name',                  # listing_name
    'host_id',
    'host_name',
    'host_tenure_years',
    'host_is_superhost',
    'host_listings_count',
    'neighbourhood_cleansed',
    'neighbourhood_group_cleansed',
    'latitude',
    'longitude',
    'property_type',
    'room_type',
    'accommodates',
    'bedrooms_imputed',
    'bathrooms_imputed',
    'price_clean',
    'price_per_bedroom',
    'minimum_nights',
    'maximum_nights',
    'availability_365',
    'occupancy_rate',
    'estimated_annual_revenue',
    'number_of_reviews',      # From original listings
    'review_scores_rating',   # From original listings
]

# Add review date columns if they exist
date_columns = []
for col in ['first_review', 'last_review']:
    if col in enriched.columns:
        date_columns.append(col)
    # Check if it might have a suffix
    elif f'{col}_reviews' in enriched.columns:
        date_columns.append(f'{col}_reviews')
    elif f'date_{col}' in enriched.columns:
        date_columns.append(f'date_{col}')

# Combine all columns
final_columns = base_columns + date_columns

print(f"\nFinal columns to select: {len(final_columns)} columns")

# Check which columns exist
missing_cols = [col for col in final_columns if col not in enriched.columns]
if missing_cols:
    print(f" Missing columns: {missing_cols}")
    # Remove missing columns
    final_columns = [col for col in final_columns if col in enriched.columns]
    print(f"Using {len(final_columns)} available columns")

# Create final DataFrame
final_df = enriched[final_columns].copy()

# Create friendly column names
new_column_names = {
    'id': 'listing_id',
    'name': 'listing_name',
    'host_id': 'host_id',
    'host_name': 'host_name',
    'host_tenure_years': 'host_tenure_years',
    'host_is_superhost': 'is_superhost',
    'host_listings_count': 'host_total_listings',
    'neighbourhood_cleansed': 'neighbourhood',
    'neighbourhood_group_cleansed': 'neighbourhood_group',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'property_type': 'property_type',
    'room_type': 'room_type',
    'accommodates': 'accommodates',
    'bedrooms_imputed': 'bedrooms',
    'bathrooms_imputed': 'bathrooms',
    'price_clean': 'price',
    'price_per_bedroom': 'price_per_bedroom',
    'minimum_nights': 'min_nights',
    'maximum_nights': 'max_nights',
    'availability_365': 'availability_365',
    'occupancy_rate': 'occupancy_rate',
    'estimated_annual_revenue': 'estimated_revenue',
    'number_of_reviews': 'review_count',
    'review_scores_rating': 'review_score',
}

# Add date column renames if they exist
if 'first_review' in final_df.columns:
    new_column_names['first_review'] = 'first_review_date'
if 'last_review' in final_df.columns:
    new_column_names['last_review'] = 'last_review_date'
if 'first_review_reviews' in final_df.columns:
    new_column_names['first_review_reviews'] = 'first_review_date'
if 'last_review_reviews' in final_df.columns:
    new_column_names['last_review_reviews'] = 'last_review_date'

# Only rename columns that exist
rename_dict = {k: v for k, v in new_column_names.items() if k in final_df.columns}
final_df = final_df.rename(columns=rename_dict)

# Clean up date columns if they exist
for date_col in ['first_review_date', 'last_review_date']:
    if date_col in final_df.columns:
        final_df[date_col] = pd.to_datetime(final_df[date_col], errors='coerce')




[6] Creating final dataset...
Available columns in enriched:
['last_scraped', 'calendar_last_scraped', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'number_of_reviews_ly', 'first_review_x', 'last_review_x', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'reviews_per_month', 'review_count', 'first_review_y', 'last_review_y']
 'first_review' not found - creating from review data

Columns after merge check:
['last_scraped', 'calendar_last_scraped', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'number_of_reviews_ly', 'first_review_x', 'last_review_x', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'reviews_per_month', 'review_count', 'first_review_y', 'last_review_y'

In [12]:
# ============================================
# 7. SAVE PROCESSED DATA
# ============================================
print("\n[7] Saving processed data...")

# Save to CSV
final_df.to_csv(PROCESSED / "listings_enriched.csv", index=False)
print(f" Saved enriched data to {PROCESSED / 'listings_enriched.csv'}")

# Save a sample for quick viewing
final_df.head(100).to_csv(PROCESSED / "listings_enriched_sample.csv", index=False)


[7] Saving processed data...
 Saved enriched data to C:\Users\User\Documents\GitHub\expernetic-data-engineer-assessment\data\processed\listings_enriched.csv


In [13]:
# ============================================
# 8. SUMMARY STATISTICS
# ============================================
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\n Lisbon Airbnb Market Overview:")
print(f"  - Total Listings: {len(final_df):,}")
print(f"  - Average Price: ${final_df['price'].mean():.2f}")
print(f"  - Median Price: ${final_df['price'].median():.2f}")
print(f"  - Average Occupancy: {final_df['occupancy_rate'].mean():.2%}")
print(f"  - Average Review Score: {final_df['review_score'].mean():.2f}")
print(f"  - Total Estimated Revenue: ${final_df['estimated_revenue'].sum():,.2f}")

print(f"\n Room Type Breakdown:")
room_type_counts = final_df['room_type'].value_counts()
for room_type, count in room_type_counts.items():
    pct = count / len(final_df) * 100
    print(f"  - {room_type}: {count:,} ({pct:.1f}%)")

print(f"\n Superhost Analysis:")
superhost_data = final_df[final_df['is_superhost'] == 't']
nonsuperhost_data = final_df[final_df['is_superhost'] != 't']

if len(superhost_data) > 0 and len(nonsuperhost_data) > 0:
    print(f"  - Superhost Listings: {len(superhost_data):,} ({len(superhost_data)/len(final_df)*100:.1f}%)")
    print(f"  - Superhost Avg Price: ${superhost_data['price'].mean():.2f}")
    print(f"  - Non-Superhost Avg Price: ${nonsuperhost_data['price'].mean():.2f}")
    price_premium = ((superhost_data['price'].mean()/nonsuperhost_data['price'].mean())-1)*100
    print(f"  - Price Premium: +{price_premium:.1f}%")

print(f"\n Occupancy Insights:")
print(f"  - Top 10% Occupancy: {final_df['occupancy_rate'].quantile(0.9):.2%}")
print(f"  - Bottom 10% Occupancy: {final_df['occupancy_rate'].quantile(0.1):.2%}")
print(f"  - Listings with >80% occupancy: {len(final_df[final_df['occupancy_rate'] > 0.8]):,}")

print(f"\n Revenue Insights:")
print(f"  - Top 10% Revenue: ${final_df['estimated_revenue'].quantile(0.9):,.2f}")
print(f"  - Average Annual Revenue: ${final_df['estimated_revenue'].mean():,.2f}")
print(f"  - Total Market Value: ${final_df['estimated_revenue'].sum():,.2f}")

# Check if date columns exist
if 'first_review_date' in final_df.columns and 'last_review_date' in final_df.columns:
    print(f"\n Review Date Range:")
    print(f"  - First review: {final_df['first_review_date'].min()}")
    print(f"  - Last review: {final_df['last_review_date'].max()}")

print(f"\n Phase 2 completed successfully!")
print(f" Data saved to: {PROCESSED / 'listings_enriched.csv'}")
print(f"\nFinal dataset shape: {final_df.shape}")
print(f"Final columns: {list(final_df.columns)}")


SUMMARY STATISTICS

 Lisbon Airbnb Market Overview:
  - Total Listings: 24,876
  - Average Price: $189.79
  - Median Price: $136.00
  - Average Occupancy: 38.70%
  - Average Review Score: 4.66
  - Total Estimated Revenue: $472,937,506.58

 Room Type Breakdown:
  - Entire home/apt: 18,444 (74.1%)
  - Private room: 6,131 (24.6%)
  - Hotel room: 153 (0.6%)
  - Shared room: 148 (0.6%)

 Superhost Analysis:
  - Superhost Listings: 7,975 (32.1%)
  - Superhost Avg Price: $201.42
  - Non-Superhost Avg Price: $183.89
  - Price Premium: +9.5%

 Occupancy Insights:
  - Top 10% Occupancy: 91.23%
  - Bottom 10% Occupancy: 3.84%
  - Listings with >80% occupancy: 4,045

 Revenue Insights:
  - Top 10% Revenue: $43,487.25
  - Average Annual Revenue: $20,893.16
  - Total Market Value: $472,937,506.58

 Review Date Range:
  - First review: 2010-07-24 00:00:00
  - Last review: 2026-07-02 00:00:00

 Phase 2 completed successfully!
 Data saved to: C:\Users\User\Documents\GitHub\expernetic-data-engineer-ass